In [30]:
# DEPENDENCIES
import numpy as np
from scipy.linalg import solve_triangular

def gauss_piv(A):
    n, _ = A.shape
    p = np.arange(n)
    for k in range(n-1):
        r = np.argmax(np.abs(A[k:, k]))
        r += k
        A[[k,r], :] = A[[r, k], :]
        p[[k,r]] = p[[r, k]]
        if np.abs(A[k, k]) < 1e-16:
            print('perno numericamente troppo piccolo')
            break
        A[k+1:, k] = A[k+1:, k] / A[k, k]
        A[k+1:, k+1:] = A[k+1:, k+1:] - \
            np.reshape(A[k+1:, k], (n-k-1, 1)) @ \
            np.reshape(A[k, k+1:], (1, n-k-1))
    return (A, p)


def solve_gauss_piv(A, b):
    A, p = gauss_piv(A)
    U = np.triu(A)
    L = np.tril(A)
    b = b[p]
    y = solve_triangular(L, b, lower=True, unit_diagonal=True)
    x = solve_triangular(U, y)    
    return x

## Trasformazioni di Householder

Le trasformazioni di Householder sono matrici simmetriche e ortogonali, utilizzate per ruotare un vettore su un asse cartesiano.

**Ricetta**: dato $a\in{\mathbb R}^n$, si definisce la matrice di Householder $U$ nel modo seguente

1. $\sigma = \|a\|$
2. $v = a + \sigma e_1$, dove $e_1$ è il primo vettore della base canonica
3. $\alpha =\frac 1 2 \|v\|^2$
4. $U = I-\frac{1}{\alpha}v v^T$

La matrice così costruita è ortogonale, simmetrica e ha la proprietà di ruotare il vettore $a$ portandolo lungo il primo asse cartesiano, ossia

$$Ua = -\sigma e_1 = \begin{pmatrix} -\|\sigma\|\\0\\\vdots\\ 0
\end{pmatrix} $$

In [31]:
n = 10
a = np.random.randn(n)

# =========================================
# Trasformazione di Householder (1ª riflessione)
# 
# Obiettivo: azzerare tutte le componenti di a
#            tranne la prima, preservando la norma
# =========================================

# 1) Norma del vettore originale
sigma = np.linalg.norm(a)
print('Norma vettore originale =', sigma)

# 2) Costruzione del vettore di Householder v
v = a.copy()

# Si usa sigma per costruire la riflessione stabile numericamente
# (evita cancellazione numerica)
v[0] += sigma

# v come vettore colonna
v = v.reshape(n, 1)

# 3) Costante di normalizzazione
alpha = (v.T @ v) / 2

# 4) Matrice di Householder
# H = I - 2 * (v v^T) / (v^T v)
H = np.eye(n) - (v @ v.T) / alpha

# =========================================
# Applicazione della trasformazione
# =========================================

print('Vettore riflesso =\n', H @ a)

# Output atteso:
# [ -||a||, 0, 0, ..., 0 ]
# (zeri numerici ~ precisione macchina)

# =========================================
# Proprietà della matrice di Householder
# =========================================

# Ortogonalità: H^T H = I
print('Ortogonalità:', np.linalg.norm(np.eye(n) - H @ H.T))

# Simmetria: H = H^T
print('Simmetria:', np.linalg.norm(H - H.T))

# Auto-inversa: H^-1 = H
print('Auto-inversa:', np.linalg.norm(H - np.linalg.inv(H)))

Norma vettore originale = 2.4865594288640946
Vettore riflesso =
 [-2.48655943e+00 -4.16333634e-17 -8.32667268e-17 -9.71445147e-17
 -4.51028104e-17 -1.12757026e-17 -1.38777878e-17 -2.77555756e-17
  0.00000000e+00  1.38777878e-17]
Ortogonalità: 3.349561673643242e-16
Simmetria: 0.0
Auto-inversa: 4.445810636145705e-16


## Complessità computazionale effettiva nel calcolo del prodotto di un vettore per una matrice di Householder

Una volta definita una trasformazione di Householder associata ad un vettore $v$ con la formula $U=I-\frac{1}{\alpha}vv^T$, se si deve calcolare un prodotto $Uy$, dove $y$ è un qualsiasi vettore in $\mathbb R^n$, l'algoritmo più efficiente si ottiene applicando prima la proprietà distributiva e poi la associativa

$$Uy =\left(I-\frac{1}{\alpha}vv^T\right)y = y-\frac{1}{\alpha}v(v^Ty)  $$

INPUT: $v,y$
1. $\alpha \leftarrow \frac{1}{2} \|v\|^2$
2. $p \leftarrow v^Ty$
3. $z \leftarrow y - v(p/\alpha)$

OUTPUT: $z=Uy$

Calcolato in questo modo, il prodotto $Uy$ ha complessità proporzionale a $n$ invece che a $n^2$

## Stabilità delle fattorizzazioni: esperimento con la matrice di Wilkinson

Consideriamo la seguente matrice $n\times n$ (matrice di Wilkinson)

$$ W = \begin{pmatrix}
1 &   & & & & & 1\\
-1& 1 & & & & & 1\\
-1&-1& 1& & & & 1\\
\vdots & & &\ddots & & &\vdots\\
-1& -1 & -1 &        && 1& 1\\
-1& -1&-1 & \cdots & \cdots &-1&1
\end{pmatrix}$$

* Scrivere una function che, dato un intero $n$, restituisce la matrice di Wilkinson di dimensione $n$.
* Costruire un problema test relativo alla matrice di Wilkinson di dimensione 10 che abbia come soluzione il vettore di tutti 1
Calcolare la soluzione del sistema con la funzione linalg.solve e confrontare la soluzione calcolata con la soluzione esatta.
* Ripetere l'esperimento con $n=60$
* Ricalcolare la matrice con $n=10$, applicare la fattorizzazione $LU$ di Gauss e stampare la matrice $U$
* Risolvere il sistema lineare iniziale ($n=60$) mediante la fattorizzazione $QR$ e confrontare con la soluzione esatta


In [32]:
import numpy as np
from scipy.linalg import lu, qr, solve_triangular

# 1) Matrice di Wilkinson
def wilkinson(n: int):
    # Triangolo inferiore con -1 sotto la diagonale
    W = np.tril(-np.ones((n, n), dtype=float), -1)
    # Aggiungo la diagonale principale (identità)
    W += np.eye(n)
    # Modifico l'ultima colonna (eccetto ultima riga)
    W[:-1, -1] = 1

    return W


# 2) Generazione del problema test
def gen_problem(n: int):
    W = wilkinson(n)     # matrice del sistema
    x = np.ones(n)       # soluzione esatta nota
    b = W @ x            # termine noto costruito a partire da x
    return W, x, b 


# 3) Risoluzione e verifica errore con n=100
n = 100

W, x, b  = gen_problem(n)

# Risoluzione del sistema lineare
s = np.linalg.solve(W, b)

# Soluzione numerica ottenuta
print('(n=100) Soluzione s =', s)

# Errore rispetto alla soluzione esatta
err_gauss = np.linalg.norm(x - s)
print('(n=100) Errore relativo =', err_gauss)


# 4) Ricalcolare la matrice con n=10, applicare la fattorizzazione LU di Gauss e stampare la matrice U
n = 10

W, b, x = gen_problem(n)

# Fattorizzazione
P, L, U = lu(W.copy())

# Stampa matrice U
print('(n=10) Matrice triangolare superiore U =\n', U)
# Matrice triangolare superiore con diagonale unitaria e l'ultima colonna con potenze di due crescenti.
# Avevamo visto che questa matrice era estremamente mal condizionata. E lo diventa sempre di più man mano che le dimensioni crescono.
# IL metodo di sostituzione all'indietro si è dimostrato instabile. La parte instabile è la fase 2.

# Verifica: PA ≈ LU
print("(n=10) Errore ||PA - LU|| =", np.linalg.norm(P @ W - L @ U))

(n=100) Soluzione s = [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 1.]
(n=100) Errore relativo = 6.782329983125268
(n=10) Matrice triangolare superiore U =
 [[  1.   0.   0.   0.   0.   0.   0.   0.   0.   1.]
 [  0.   1.   0.   0.   0.   0.   0.   0.   0.   2.]
 [  0.   0.   1.   0.   0.   0.   0.   0.   0.   4.]
 [  0.   0.   0.   1.   0.   0.   0.   0.   0.   8.]
 [  0.   0.   0.   0.   1.   0.   0.   0.   0.  16.]
 [  0.   0.   0.   0.   0.   1.   0.   0.   0.  32.]
 [  0.   0.   0.   0.   0.   0.   1.   0.   0.  64.]
 [  0.   0.   0.   0.   0.   0.   0.   1.   0. 128.]
 [  0.   0.   0.   0.   0.   0.   0.   0.   1. 256.]
 [  0.   0.   0.   0.   0.   0.   0.   0.   0. 512.]]
(n=10) Errore ||PA - LU|| = 0.0


In [ ]:
# 5) Implementiamo la soluzione del sistema migliore post fattorizzazione QR

n = 10

W, b, x = gen_problem(n)

# Calc sol
s = np.linalg.solve(W, b)

# Fattorizzazione QR
Q, R = qr(W)

# Generazione Termini Noti
y = Q.T @ b

# Risoluzione vettore incognito
x_QR = solve_triangular(R, y)

# Errore relativo Gauss vs QR
print('(n=100) Errore relativo (Gauss) =', err_gauss)
print('(n=100) Errore relativo (QR) =', np.linalg.norm(x_QR - s))  # Poco sopra prec di macchina, approx 0


(n=100) Errore relativo (Gauss) = 6.782329983125268
(n=100) Errore relativo (QR) = 3.4241679216103947e-16
